In [7]:
%pip install numpy 
%pip install pandas 
%pip install torch 
%pip install scikit-learn
%pip install tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Using cached torch-2.7.1-cp313-cp313-win_amd64.whl.metadata (28 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.7.1-cp313-cp313-win_amd64.whl (216.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached networkx-3.5-py3-none-any.whl (2.0 MB)

   ---------------------------------------- 0/5 [mpmath]
   ---------------------------------------- 0/5 [mpmath]
   ----------------------------------------

In [ ]:
import torch

print(torch.__version__)

2.7.1+cpu


In [ ]:
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# === Load player metadata ===
main_csv = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_ratings_original_updated.csv"
stats_folder = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_fbref_tables"
df = pd.read_csv(main_csv)[:10000]

# Preserve original player names before encoding
df["raw_name"] = df["Name"]

# Drop irrelevant columns
df.drop(
    columns=["Unnamed: 0", "Player URL", "Team Link", "fbref_url", "fbref_alltimestat"],
    inplace=True,
)

# === Encode categorical features ===
cat_cols = ["Name", "Team", "Positions", "Nationality"]
encoders = {col: LabelEncoder().fit(df[col].astype(str)) for col in cat_cols}
for col in cat_cols:
    df[col] = encoders[col].transform(df[col].astype(str))

# === Normalize numeric features ===
num_cols = ["Age"]
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# === Target columns ===
target_cols = ["Rating", "Potential", "Value"]
y = df[target_cols].values


# === Aggregation function for numeric stats + metadata ===
def load_player_stats(folder_path, player_id, name):
    folder_name = f"{name}_{player_id}"
    full_path = os.path.join(folder_path, folder_name)
    if not os.path.isdir(full_path):
        print(f"Missing folder: {full_path}")
        return pd.Series(dtype=object)

    summaries = []
    for file in os.listdir(full_path):
        if file.endswith(".csv"):
            try:
                file_path = os.path.join(full_path, file)
                data = pd.read_csv(file_path)

                summary = {}

                # Numeric columns: take mean
                numeric = data.select_dtypes(include=np.number)
                for col in numeric.columns:
                    summary[f"{file}_{col}"] = numeric[col].mean()

                # Metadata columns: grab most common values
                for meta_col in ["season", "club", "date", "team", "competition"]:
                    if meta_col in data.columns:
                        summary[f"{file}_{meta_col}"] = data[meta_col].mode().iloc[0]

                summaries.append(pd.Series(summary))
            except Exception as e:
                print(f"Failed to read {file}: {e}")
                continue

    if summaries:
        return pd.concat(
            summaries, axis=1
        ).T.mean()  # Average numeric values, retain representative metadata
    else:
        return pd.Series(dtype=object)


# === Build aggregated stats dataframe ===
print("Aggregating per-player stats...")
stats_rows = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    stats = load_player_stats(stats_folder, row["player_id"], row["raw_name"])
    stats_rows.append(stats)

stats_df = pd.DataFrame(stats_rows).fillna(0)
df_combined = pd.concat(
    [df.reset_index(drop=True), stats_df.reset_index(drop=True)], axis=1
)
# Drop targets, make a copy to avoid modifying original
X_df = df_combined.drop(columns=target_cols).copy()

# Encode all object-type columns
for col in X_df.select_dtypes(include=["object"]).columns:
    X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))

# Now convert to numpy array
X = X_df.values
print("Finished preprocessing and feature aggregation!")

Aggregating per-player stats...


  0%|          | 29/10000 [00:58<5:55:52,  2.14s/it]

Failed to read stats_shooting_collapsed.csv: No columns to parse from file


 10%|▉         | 980/10000 [25:53<3:58:35,  1.59s/it]

Missing folder: C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_fbref_tables\Franck Honorat_nan


...

 98%|█████████▊| 9796/10000 [2:34:04<01:45,  1.94it/s]

Missing folder: C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_fbref_tables\M. Bundgaard_nan


 99%|█████████▊| 9854/10000 [2:34:41<01:08,  2.14it/s]

Missing folder: C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_fbref_tables\Ran Binyamin_nan


100%|██████████| 10000/10000 [2:36:00<00:00,  1.07it/s]


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Scale targets (y)
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)


class PlayerDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = PlayerDataset(X_train, y_train)
val_dataset = PlayerDataset(X_test, y_test)
# === Dataloaders ===
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    ),
    batch_size=128,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32),
    ),
    batch_size=128,
)


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=dim, num_heads=heads, batch_first=True
        )
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class ResNetAttentionModel(nn.Module):
    def __init__(self, input_dim, output_dim, projected_dim=256, heads=4, dropout=0.2):
        super().__init__()
        assert projected_dim % heads == 0, (
            "projected_dim must be divisible by number of attention heads"
        )

        # Initial projection
        self.project = nn.Sequential(
            nn.Linear(input_dim, projected_dim), nn.LayerNorm(projected_dim), nn.ReLU()
        )

        # Residual block with skip connection
        self.res_block = nn.Sequential(
            nn.Linear(projected_dim, projected_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(projected_dim, projected_dim),
            nn.LayerNorm(projected_dim),
        )

        # Transformer block for contextual modeling
        self.transformer = TransformerBlock(projected_dim, heads=heads)

        # Attention mechanism
        self.attn = nn.Sequential(
            nn.Linear(projected_dim, 64), nn.Tanh(), nn.Linear(64, 1), nn.Softmax(dim=1)
        )

        # Final prediction head
        self.head = nn.Sequential(
            nn.Linear(projected_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
        )

    def forward(self, x):
        x = self.project(x)
        x = self.res_block(x) + x  # residual connection
        x = x.unsqueeze(1)  # add sequence dimension
        x = self.transformer(x)
        weights = self.attn(x)
        x = torch.sum(weights * x, dim=1)  # attention-weighted sum
        return self.head(x)

In [ ]:
# === Model, Optimizer, Criterion ===
model = ResNetAttentionModel(
    input_dim=X.shape[1], output_dim=y.shape[1], projected_dim=256
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)
criterion = nn.MSELoss()

# === Training Loop with Early Stopping ===
best_val_loss = float("inf")
patience = 10
counter = 0

for epoch in range(1, 2001):
    model.train()
    train_loss = 0
    train_loop = tqdm(train_loader, desc=f"[Epoch {epoch}] Train", leave=False)
    for batch_x, batch_y in train_loop:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_loop.set_postfix(batch_loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f" Epoch {epoch:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}"
    )

    # === Early Stopping Logic ===
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("⏹️ Early stopping triggered.")
            break


[Epoch 1] Train:   0%|          | 0/7 [00:00<?, ?it/s]

 Epoch 001 | Train Loss: 1.0071 | Val Loss: 0.6457


 Epoch 002 | Train Loss: 0.7413 | Val Loss: 0.5981


 Epoch 003 | Train Loss: 0.6171 | Val Loss: 0.5772


 Epoch 004 | Train Loss: 0.4365 | Val Loss: 0.5642


...

 Epoch 011 | Train Loss: 0.1856 | Val Loss: 0.7309


 Epoch 012 | Train Loss: 0.1702 | Val Loss: 0.7577


 Epoch 013 | Train Loss: 0.1441 | Val Loss: 0.7608


 Epoch 014 | Train Loss: 0.1632 | Val Loss: 0.8095


 Epoch 015 | Train Loss: 0.1378 | Val Loss: 0.8312
⏹️ Early stopping triggered.
